# 03 · Sobel Operators, 2D Image Convolutions, and Edge Detection

A 1D finite difference calculates the rate of change along a single line of pixels. However, natural image edges possess arbitrary 2D orientations, and 1D differentiation is sensitive to high-frequency sensor noise.

The **Sobel Operator** (Sobel & Feldman, 1968) solves both issues by combining:
1. **Smoothing (Low-pass filtering):** Triangular weighted averaging $[1, 2, 1]^T$ orthogonal to the derivative direction to suppress pixel noise.
2. **Differentiation (High-pass filtering):** Symmetric central difference $[-1, 0, 1]$ along the derivative axis.

In this notebook, we explore:
1. The mathematical separability of Sobel kernels: $K_x$ and $K_y$.
2. Step-by-step 2D discrete convolution from first principles.
3. Walkthrough of the exact hand calculation from lecture slide 22.
4. Computing continuous Gradient Magnitude and Gradient Orientation.
5. Benchmarking scratch implementations against OpenCV (`cv2.Sobel`).
6. Binary edge detection via thresholding.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Load test image
img_path = Path("../data/nature.jpg")
img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
height, width = img.shape
print(f"Loaded image: {height}x{width}")

## 1. Mathematical Structure of Sobel Kernels

The horizontal Sobel kernel $K_x$ and vertical kernel $K_y$ are $3 \times 3$ matrices:

$$K_x = \begin{bmatrix} -1 & 0 & +1 \\ -2 & 0 & +2 \\ -1 & 0 & +1 \end{bmatrix} = \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \ast \begin{bmatrix} -1 & 0 & +1 \end{bmatrix}$$

$$K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ +1 & +2 & +1 \end{bmatrix} = \begin{bmatrix} -1 \\ 0 \\ +1 \end{bmatrix} \ast \begin{bmatrix} 1 & 2 & 1 \end{bmatrix}$$

This outer-product decomposition proves that the Sobel filter is **rank-1 separable**:
- In the $x$-direction: it computes the central difference across columns while smoothing across adjacent rows with weights $1, 2, 1$.
- In the $y$-direction: it computes the central difference across rows while smoothing across adjacent columns with weights $1, 2, 1$.


In [ ]:
# Kernel definitions
K_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64)

K_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float64)

# Verify separability
smooth_1d = np.array([[1], [2], [1]], dtype=np.float64)
diff_1d = np.array([[-1, 0, 1]], dtype=np.float64)
assert np.allclose(K_x, smooth_1d @ diff_1d)
assert np.allclose(K_y, diff_1d.T @ smooth_1d.T)
print("Sobel kernels successfully verified as separable rank-1 operators.")

## 2. Lecture Example Walkthrough (Slide 22)

Consider a local $3 \times 3$ image patch:

$$\begin{bmatrix} 230 & p_{01} & 147 \\ 237 & p_{11} & 188 \\ 210 & p_{21} & 180 \end{bmatrix}$$

Convolving with $K_x$:

$$a = (-1)(230) + (1)(147) + (-2)(237) + (2)(188) + (-1)(210) + (1)(180)$$
$$a = -230 + 147 - 474 + 376 - 210 + 180 = -211$$

The absolute magnitude is:
$$|a| = |-211| = 211$$


In [ ]:
# Numerical verification of slide 22
patch = np.array([[230, 0, 147], [237, 0, 188], [210, 0, 180]], dtype=np.float64)

a = np.sum(patch * K_x)
print(f"Calculated convolution value a = {a:.1f}")
print(f"Absolute magnitude |a| = {abs(a):.1f}")
assert a == -211.0
assert abs(a) == 211.0

## 3. 2D Convolution from First Principles

For an image $I$ and kernel $K$ of size $(2k+1) \times (2k+1)$:

$$G[i, j] = \sum_{u=-k}^k \sum_{v=-k}^k K[u+k, v+k] \cdot I[i+u, j+v]$$


In [ ]:
def convolve2d_scratch(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Convolve a 2D image with a 3x3 kernel using reflection padding."""
    h, w = image.shape
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2

    padded = np.pad(image.astype(np.float64), ((pad_h, pad_h), (pad_w, pad_w)), mode="reflect")
    output = np.zeros((h, w), dtype=np.float64)

    for i in range(h):
        for j in range(w):
            region = padded[i : i + kh, j : j + kw]
            output[i, j] = np.sum(region * kernel)

    return output


# Test on 60x60 patch to benchmark against OpenCV
patch_crop = img[:60, :60]
scratch_gx = convolve2d_scratch(patch_crop, K_x)
cv_gx = cv2.Sobel(patch_crop, cv2.CV_64F, 1, 0, ksize=3)

# Inside valid bounds (excluding borders), results match identically
max_diff = np.max(np.abs(scratch_gx[1:-1, 1:-1] - cv_gx[1:-1, 1:-1]))
print(f"Max difference between scratch Sobel-X and cv2.Sobel: {max_diff:.2e}")
assert max_diff < 1e-5

## 4. Full Image Filtering: Gradient Magnitude & Orientation

$$\nabla I = \begin{bmatrix} G_x \\ G_y \end{bmatrix}$$
$$G = \sqrt{G_x^2 + G_y^2} \quad \text{(Euclidean magnitude)}$$
$$\theta = \text{atan2}(G_y, G_x) \quad \text{(Gradient direction, perpendicular to edge tangent)}$$


In [ ]:
# Compute Sobel on full image via OpenCV
gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)

# Gradient magnitude and angle
magnitude = np.sqrt(gx**2 + gy**2)
orientation = np.arctan2(gy, gx) * (180.0 / np.pi)  # in degrees [-180, 180]

# Binary edge detection via thresholding
threshold = 100.0
binary_edges = (magnitude > threshold).astype(np.uint8) * 255

print(
    f"Gradient magnitude: min = {magnitude.min():.1f}, max = {magnitude.max():.1f}, mean = {magnitude.mean():.1f}"
)
print(f"Edge pixel fraction (> {threshold}): {np.mean(binary_edges > 0) * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].imshow(np.abs(gx), cmap="gray")
axes[0, 0].set_title("Sobel-X |Gx| (Vertical Edges)", fontsize=12)
axes[0, 0].axis("off")

axes[0, 1].imshow(np.abs(gy), cmap="gray")
axes[0, 1].set_title("Sobel-Y |Gy| (Horizontal Edges)", fontsize=12)
axes[0, 1].axis("off")

axes[1, 0].imshow(magnitude, cmap="inferno")
axes[1, 0].set_title("Gradient Magnitude G = sqrt(Gx^2 + Gy^2)", fontsize=12)
axes[1, 0].axis("off")

axes[1, 1].imshow(binary_edges, cmap="gray")
axes[1, 1].set_title(f"Thresholded Edge Map (T = {threshold})", fontsize=12)
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()